In [3]:
from datasets import load_dataset

# HaluEval gives you exactly the triples you need:
# document (context), hallucinated summary, correct summary
dataset = load_dataset("pminervini/HaluEval", "summarization")

print(dataset)
example = dataset['data'][0]
print(example)
# Look at the actual field names — they vary by HaluEval version
# Usually: 'document', 'hallucinated_summary', 'right_summary'

DatasetDict({
    data: Dataset({
        features: ['document', 'right_summary', 'hallucinated_summary'],
        num_rows: 10000
    })
})
{'document': 'Marseille, France (CNN)The French prosecutor leading an investigation into the crash of Germanwings Flight 9525 insisted Wednesday that he was not aware of any video footage from on board the plane. Marseille prosecutor Brice Robin told CNN that "so far no videos were used in the crash investigation." He added, "A person who has such a video needs to immediately give it to the investigators." Robin\'s comments follow claims by two magazines, German daily Bild and French Paris Match, of a cell phone video showing the harrowing final seconds from on board Germanwings Flight 9525 as it crashed into the French Alps. All 150 on board were killed. Paris Match and Bild reported that the video was recovered from a phone at the wreckage site. The two publications described the supposed video, but did not post it on their websites. The publica

In [4]:
def format_for_correction(example):
    """
    Format each example into the input/target structure BART needs.
    
    Input : "context: {doc} [SEP] summary: {hallucinated}"
    Target: "{correct_summary}"
    
    This is identical in spirit to how you formatted T5 inputs 
    in Module 6 with "summarize: " prefix — just a different prefix
    structure since we have TWO pieces of input (context + summary).
    """
    input_text = (
        f"context: {example['document']} "
        f"[SEP] summary: {example['hallucinated_summary']}"
    )
    target_text = example['right_summary']
    
    return {
        "input_text": input_text,
        "target_text": target_text
    }

formatted_dataset = dataset['data'].map(format_for_correction)
print(formatted_dataset[0]['input_text'][:200])
print(formatted_dataset[0]['target_text'][:200])

context: Marseille, France (CNN)The French prosecutor leading an investigation into the crash of Germanwings Flight 9525 insisted Wednesday that he was not aware of any video footage from on board the
Marseille prosecutor says "so far no videos were used in the crash investigation" despite media reports. Journalists at Bild and Paris Match are "very confident" the video clip is real, an editor says


In [5]:
from transformers import AutoTokenizer

model_name = "t5-small"
tokenizer  = AutoTokenizer.from_pretrained(model_name)

MAX_INPUT_LENGTH  = 256   # context can be long — keep tight for VRAM
MAX_TARGET_LENGTH = 64    # corrected sentence, not full document

def tokenize_correction(examples):
    """
    Exact same pattern as your T5 summarization tokenization 
    in Module 6, Lesson 6.1 — input/target tokenized separately.
    """
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    label_ids = [
        [t if t != tokenizer.pad_token_id else -100 for t in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs


tokenized_dataset = formatted_dataset.map(
    tokenize_correction,
    batched=True,
    remove_columns=formatted_dataset.column_names
)
tokenized_dataset.set_format("torch")

split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

Train: 9000 | Val: 1000


In [6]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Load BART normally (small enough not to need 4-bit quantization)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ── LoRA config ──
# Same structure as Phase 3, but task_type changes:
# Phase 3 used CAUSAL_LM (decoder-only, like Qwen)
# Here we use SEQ_2_SEQ_LM (encoder-decoder, like BART)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,   # ← key difference from Phase 3
    target_modules=["q", "v"],  
    # BART's attention uses q_proj/v_proj naming
    # (Qwen used q_proj/k_proj/v_proj/o_proj — check your model's
    #  actual layer names if this errors: print(model) to see them)
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 6550.24it/s]
Could not find the bitsandbytes CUDA binary at WindowsPath('c:/Users/acer/AppData/Local/Programs/Python/Python311/Lib/site-packages/bitsandbytes/libbitsandbytes_cuda126.dll')
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654


In [7]:
from transformers import (
    Seq2SeqTrainingArguments, Trainer, DataCollatorForSeq2Seq
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_lora_corrector",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16, fits 8GB VRAM
    learning_rate=1e-4,              # LoRA can use higher LR than full fine-tune
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=64,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("🚀 Training hallucination corrector...")
trainer.train()

# Save only the LoRA adapter (tiny file, a few MB)
model.save_pretrained("./bart_lora_corrector_final")
tokenizer.save_pretrained("./bart_lora_corrector_final")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Training hallucination corrector...


c:\Users\acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\data\data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,19.113954,2.177184
2,19.383461,2.135743
3,18.693925,2.121435
4,18.826654,2.112707
5,19.064934,2.111303


('./bart_lora_corrector_final\\tokenizer_config.json',
 './bart_lora_corrector_final\\tokenizer.json')

In [8]:
def correct_hallucination(context: str, hallucinated: str, 
                          model, tokenizer, device) -> str:
    """
    Same generation pattern as Module 6 summarize()/translate() functions —
    just a different input format and task.
    """
    model.eval()
    
    input_text = f"context: {context} [SEP] summary: {hallucinated}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,
        truncation=True
    ).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Test it
test_context = "Nepal's capital is Kathmandu, located at 1400 meters altitude."
test_hallucinated = "Nepal's capital is Pokhara, located at 2000 meters altitude."

corrected = correct_hallucination(
    test_context, test_hallucinated, model, tokenizer, device
)

print(f"Context     : {test_context}")
print(f"Hallucinated: {test_hallucinated}")
print(f"Corrected   : {corrected}")

Context     : Nepal's capital is Kathmandu, located at 1400 meters altitude.
Hallucinated: Nepal's capital is Pokhara, located at 2000 meters altitude.
Corrected   : Nepal's capital is Kathmandu, located at 1400 meters altitude.
